In [2]:
import os

os.chdir("../")

In [3]:
from src.datasets import data_importation, data_preprocessing
from src.models import torch_model

/home/onyxia/work/LabelGuard/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import duckdb

con = duckdb.connect()

In [5]:
df_real = data_importation.fetch_original_data(
    path="s3://mateom/graal/embeddings/NAF2025/original_train_cleaned.parquet",
    n_samples=1000,
    random_state=42
)

In [6]:
code_list = df_real["code"].to_list()

In [7]:
df_synth = data_importation.select_synthetic_data(
    path="s3://mateom/graal/embeddings/NAF2025/2026-03-16_gemma_synth.parquet",
    code_list=code_list,
    random_state=42
)

In [8]:
X_train, X_test, y_train, y_test = data_preprocessing.create_train_test(df_real, df_synth, train_size=0.7, random_state=42)

In [9]:
import numpy as np
import torch

X_train = torch.from_numpy(X_train.astype(np.float32))
y_train = torch.from_numpy(y_train.astype(np.float32))
X_test = torch.from_numpy(X_test.astype(np.float32))
y_test = torch.from_numpy(y_test.astype(np.float32))

In [10]:
model = torch_model.TorchMLPClassifier(
    input_dim=4096,
    hidden_layers=[128, 64],
)

In [11]:
model.fit(X_train, y_train, epochs=3)

Epoch 1/3 - loss: 0.5212
Epoch 2/3 - loss: 0.2535
Epoch 3/3 - loss: 0.1314


In [12]:
from sklearn.metrics import accuracy_score, f1_score

In [13]:
y_pred = model.predict(X_test)

In [14]:
print(accuracy_score(y_pred=y_pred, y_true=y_test))

print(f1_score(y_pred=y_pred, y_true=y_test))

0.825
0.8264462809917356


In [15]:
y_pred_train = model.predict(X_train)

In [16]:
print(accuracy_score(y_pred=y_pred_train, y_true=y_train))

print(f1_score(y_pred=y_pred_train, y_true=y_train))

0.9857142857142858
0.9858956276445698


In [17]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression()

clf.fit(X_train, y_train)

/home/onyxia/work/LabelGuard/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:1352: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [18]:
y_pred = clf.predict(X_test)

print(accuracy_score(y_pred=y_pred, y_true=y_test))

print(f1_score(y_pred=y_pred, y_true=y_test))

0.845
0.8421052631578947


In [34]:
import xgboost as xgb

n_estimators = 20
learning_rate = 0.5
max_depth = 3
min_split_loss = 0
subsample = 0.7
colsample_bytree = 0.3
random_state = 42

xgb_model = xgb.XGBClassifier(
            n_estimators=n_estimators,
            eta=learning_rate,
            gamma=min_split_loss,
            max_depth=max_depth,
            seed=random_state,
            subsample=subsample,
            colsample_bytree=colsample_bytree
        )

xgb_model.fit(X_train.numpy(), y_train.numpy())

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.3
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes f

In [35]:
y_pred = xgb_model.predict(X_test)

print(accuracy_score(y_pred=y_pred, y_true=y_test))

print(f1_score(y_pred=y_pred, y_true=y_test))

0.7266666666666667
0.7275747508305648


In [36]:
y_pred = xgb_model.predict(X_train)

print(accuracy_score(y_pred=y_pred, y_true=y_train))

print(f1_score(y_pred=y_pred, y_true=y_train))

0.9692857142857143
0.9692197566213314


In [40]:
xgb_model.predict_proba(X_test)[:, 1].reshape(-1, 1)

array([[0.06369402],
       [0.48675498],
       [0.91733396],
       [0.03218282],
       [0.45340395],
       [0.90135306],
       [0.03152507],
       [0.7573209 ],
       [0.9361248 ],
       [0.19370605],
       [0.18319632],
       [0.97654617],
       [0.9968303 ],
       [0.88667387],
       [0.13872461],
       [0.6219027 ],
       [0.1242747 ],
       [0.13851951],
       [0.8096676 ],
       [0.3571492 ],
       [0.72141105],
       [0.97036695],
       [0.9933369 ],
       [0.06331553],
       [0.19486225],
       [0.42431974],
       [0.9382819 ],
       [0.18276075],
       [0.8258891 ],
       [0.04275453],
       [0.5409465 ],
       [0.2571189 ],
       [0.18653807],
       [0.2597597 ],
       [0.9753896 ],
       [0.1328456 ],
       [0.93909407],
       [0.4546776 ],
       [0.59968185],
       [0.23435275],
       [0.18702935],
       [0.55955386],
       [0.07737518],
       [0.4008955 ],
       [0.90711826],
       [0.07627035],
       [0.02956703],
       [0.791